In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

print("Libraries imported successfully!")

In [ ]:
# Simulating historical IPL match combinations for training
np.random.seed(42)
n_samples = 1000

teams = [
    "Mumbai Indians", "Chennai Super Kings", "Royal Challengers Bengaluru",
    "Gujarat Titans", "Lucknow Super Giants", "Rajasthan Royals",
    "Kolkata Knight Riders", "Sunrisers Hyderabad", "Delhi Capitals", "Punjab Kings"
]

# Standard historical win coefficients to make synthetic data realistic
team_stats = {
    "Mumbai Indians": 0.55, "Chennai Super Kings": 0.58, "Royal Challengers Bengaluru": 0.48,
    "Gujarat Titans": 0.60, "Lucknow Super Giants": 0.52, "Rajasthan Royals": 0.51,
    "Kolkata Knight Riders": 0.53, "Sunrisers Hyderabad": 0.49, "Delhi Capitals": 0.45, "Punjab Kings": 0.44
}

data = []
for _ in range(n_samples):
    t1, t2 = np.random.choice(teams, size=2, replace=False)
    t1_toss_pct = team_stats[t1]
    t2_toss_pct = team_stats[t2]
    
    # Simulate toss winner (1 if Team 1 wins toss, 0 if Team 2 wins)
    toss_prob = t1_toss_pct / (t1_toss_pct + t2_toss_pct)
    team1_toss = 1 if np.random.random() < toss_prob else 0
    
    # Simulate match winner influenced by both team strengths and the toss outcome
    match_prob = (t1_toss_pct * 0.6 + t1_toss * 0.1) / (t1_toss_pct * 0.6 + t2_toss_pct * 0.6 + 0.1)
    match_team1 = 1 if np.random.random() < match_prob else 0
    
    data.append({
        'team1': t1, 'team2': t2,
        'team1_toss_pct': t1_toss_pct, 'team2_toss_pct': t2_toss_pct,
        'team1_toss': team1_toss, 'match_team1': match_team1
    })

df = pd.DataFrame(data)
print(f"Dataset generated with {df.shape[0]} match history records.")
df.head()

In [ ]:
# Features for Toss Model
X_toss = df[['team1_toss_pct', 'team2_toss_pct']]
y_toss = df['team1_toss']

# Features for Match Model (includes who won the toss)
X_match = df[['team1_toss', 'team1_toss_pct', 'team2_toss_pct']]
y_match = df['match_team1']

# Split both into Train and Test splits (80% train, 20% test)
X_train_toss, X_test_toss, y_train_toss, y_test_toss = train_test_split(X_toss, y_toss, test_size=0.2, random_state=42)
X_train_match, X_test_match, y_train_match, y_test_match = train_test_split(X_match, y_match, test_size=0.2, random_state=42)

print("Data successfully split into training and testing sets.")

In [ ]:
toss_model = LogisticRegression()
toss_model.fit(X_train_toss, y_train_toss)

toss_preds = toss_model.predict(X_test_toss)
toss_acc = accuracy_score(y_test_toss, toss_preds)

print(f"Toss Prediction Model Accuracy: {toss_acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_toss, toss_preds))

In [ ]:
match_model = LogisticRegression()
match_model.fit(X_train_match, y_train_match)

match_preds = match_model.predict(X_test_match)
match_acc = accuracy_score(y_test_match, match_preds)

print(f"Match Prediction Model Accuracy: {match_acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_match, match_preds))

In [ ]:
print("Starting model export process...")

# Ensure the target directory exists
os.makedirs('models', exist_ok=True)

# Export the Toss Model
joblib.dump(toss_model, 'models/toss_model.pkl')
print("✅ Successfully saved toss_model to 'models/toss_model.pkl'")

# Export the Match Model
joblib.dump(match_model, 'models/match_model.pkl')
print("✅ Successfully saved match_model to 'models/match_model.pkl'")

print("\nAll models ready for your FastAPI backend application!")